# Exploring BV-BRC data by pathogen 
This notebook will analyze the distributions of BV-BRC data by grouping them by target pathogens. The target pathogens are defined as ESKAPE pathogens combined with a couple other high-profile pathogens with diverse environments and public health implications

First we need to pull together the NCBI taxonomy along with parent tax ids so we can identify all tax id's belonging to each pathogen organism

In [20]:
import pandas as pd

tax_names_filepath = r"C:\Users\Clayg\OneDrive\Desktop\RaviLab\Codeathon\bvbrc-goldstd-metadata\data\names.dmp"

tax_names_df = pd.read_csv(
    tax_names_filepath,
    sep=r"\t\|\t?",
    engine="python",
    header=None
)

tax_nodes_filepath = r"C:\Users\Clayg\OneDrive\Desktop\RaviLab\Codeathon\bvbrc-goldstd-metadata\data\nodes.dmp"

tax_nodes_df = pd.read_csv(
    tax_nodes_filepath,
    sep=r"\t\|\t?",
    engine="python",
    header=None
)

In [21]:
#Ensure standard names for columns
tax_nodes_df.rename(columns={0:"tax_id", 1: "parent_id"}, inplace=True)
tax_names_df.rename(columns={0:'tax_id', 1:'name'}, inplace=True)


In [22]:
#Combine taxonomy names with parent ids
df_merged = tax_names_df.merge(
    tax_nodes_df[['tax_id', 'parent_id']], 
    on='tax_id', 
    how='left'
)

In [23]:
#Inititate the pathogen dictionary

pathogen_dict = {"Enterococcus faecium":[1352], 
                 "Staphylococcus aureus":[1280], 
                 "Klebsiella pneumoniae":[573], 
                 "Acinetobacter baumannii":[470], 
                 "Pseudomonas aeruginosa":[287], 
                 "Enterobacter":[547]
                 }


Now we want to create a list of all child tax id's for each pathogen so we can group them all under one label 

In [24]:
from collections import defaultdict

def get_pathogen_descendants_top_down(pathogen_dict, taxonomy_df):
    # Build a dictionary mapping parent -> list of all its children
    children_map = defaultdict(list)
    for tax_id, parent_id in zip(taxonomy_df['tax_id'], taxonomy_df['parent_id']):
        if tax_id != parent_id: # Ignore self-loops
            children_map[parent_id].append(tax_id)
            
    # For each pathogen, traverse downward to find all descendants
    for pathogen, ids in pathogen_dict.items():
        root_id = ids[0]
        queue = [root_id]
        
        while queue:
            curr = queue.pop(0)
            children = children_map.get(curr, [])
            
            for child in children:
                if child not in ids:
                    ids.append(child)
                    queue.append(child)
                    
    return pathogen_dict

In [25]:
pathogen_list_cmplt = get_pathogen_descendants_top_down(pathogen_dict, df_merged)

In [26]:
#import the BV-BRC bacterial dataset

filepath = r"C:\Users\Clayg\OneDrive\Desktop\RaviLab\Codeathon\bvbrc-goldstd-metadata\data\bvbrc_clean.csv"

bv_brc_df = pd.read_csv(filepath)

C:\Users\Clayg\AppData\Local\Temp\ipykernel_5004\4137996921.py:5: DtypeWarning: Columns (0: genome.additional_metadata, 1: genome.altitude, 2: genome.antimicrobial_resistance, 3: genome.antimicrobial_resistance_evidence, 4: genome.assembly_accession, 5: genome.assembly_method, 6: genome.authors, 7: genome.bioproject_accession, 8: genome.biosample_accession, 9: genome.biovar, 10: genome.body_sample_site, 11: genome.body_sample_subsite, 12: genome.cell_shape, 13: genome.class, 14: genome.coarse_consistency, 15: genome.collection_date, 16: genome.collection_date_dr, 17: genome.comments, 18: genome.completion_date, 19: genome.contig_l50, 20: genome.culture_collection, 21: genome.depth, 22: genome.disease, 23: genome.family, 24: genome.fine_consistency, 25: genome.genbank_accessions, 26: genome.genetic_code, 27: genome.genome_quality_flags, 28: genome.genus, 29: genome.geographic_group, 30: genome.geographic_location, 31: genome.gram_stain, 32: genome.habitat, 33: genome.host_age, 34: genom

In [27]:
#Identify relevant tax id column in bv-brc data

bv_cols = bv_brc_df.columns.to_list()

tax_cols = [col for col in bv_cols if "tax" in col.lower()]
tax_cols

['genome.taxon_id',
 'genome.taxon_lineage_ids',
 'genome.taxon_lineage_names',
 'genome.taxonomy']

For each biosample, we'll assign a pathogen name if its tax id is a child of one of our pathogens. We will ignore those that don't match any of our pathogens

In [28]:
import numpy as np

# 1. Invert the dictionary to create a flat lookup: {tax_id: pathogen_name}
tax_to_pathogen = {
    tax_id: pathogen
    for pathogen, tax_ids in pathogen_list_cmplt.items()
    for tax_id in tax_ids
}

# 2. (Optional safeguard) Ensure taxon IDs are comparable types.
# BV-BRC data sometimes imports IDs as floats (due to NaNs) or strings.
# If needed, align the column type to integers (handling missing values cleanly):
bv_brc_df["genome.taxon_id"] = pd.to_numeric(
    bv_brc_df["genome.taxon_id"], errors="coerce"
).astype("Int64")

# 3. Map the pathogen name; unmapped IDs default directly to NaN
bv_brc_df["pathogen"] = bv_brc_df["genome.taxon_id"].map(tax_to_pathogen)


In [29]:
print(bv_brc_df["pathogen"].value_counts())
prct_nan = sum(bv_brc_df["pathogen"].isna())/len(bv_brc_df)
print(f"{round(prct_nan, 3)*100}% of samples don't belong to a target pathogen")


pathogen
Klebsiella pneumoniae      54467
Staphylococcus aureus      25115
Acinetobacter baumannii    21738
Pseudomonas aeruginosa     16340
Enterococcus faecium        9795
Enterobacter                7682
Name: count, dtype: int64
90.2% of samples don't belong to a target pathogen


In [30]:
#Remove samples without a target pathogen

bv_brc_df = bv_brc_df[~bv_brc_df['pathogen'].isna()]

In [31]:
#Export as a parquet file

bv_brc_to_parq = bv_brc_df.copy()

# Convert object and category columns to standard string
object_cols = bv_brc_to_parq.select_dtypes(include=['object', 'category']).columns
bv_brc_to_parq[object_cols] = bv_brc_to_parq[object_cols].astype(str)

#save to parquet
output_path = r"C:\Users\Clayg\OneDrive\Desktop\RaviLab\Codeathon\bvbrc-goldstd-metadata\data\bvbrc_eskape.parquet"
bv_brc_to_parq.to_parquet(output_path, engine="pyarrow", index=False)

C:\Users\Clayg\AppData\Local\Temp\ipykernel_5004\858354105.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = bv_brc_to_parq.select_dtypes(include=['object', 'category']).columns


In [32]:
bv_brc_to_parq.head()

,genome.additional_metadata,genome.altitude,genome.antimicrobial_resistance,genome.antimicrobial_resistance_evidence,genome.assembly_accession,genome.assembly_method,genome.authors,genome.bioproject_accession,genome.biosample_accession,genome.biovar,...,genome.taxon_lineage_ids,genome.taxon_lineage_names,genome.taxonomy,genome.temperature_range,genome.text,genome.trna,genome.type_strain,genome.user_read,genome.user_write,pathogen
7,NaN,NaN,NaN,NaN,GCA_000220025.2,NaN,NaN,PRJNA64619,SAMN02471202,NaN,...,131567::2::3379134::1224::1236::72274::135621:...,cellular organisms::Bacteria::Pseudomonadati::...,cellular organisms; Bacteria; Pseudomonadati; ...,NaN,NaN,44.0,NaN,NaN,NaN,Pseudomonas aeruginosa
1547,NaN,NaN,NaN,NaN,GCA_000784385.1,ClC Genomics Workbench v. 3.2.0,NaN,PRJNA65467,SAMN03196741,NaN,...,131567::2::1783272::1239::91061::1385::90964::...,cellular organisms::Bacteria::Bacillati::Bacil...,cellular organisms; Bacteria; Bacillati; Bacil...,NaN,NaN,17.0,NaN,NaN,NaN,Staphylococcus aureus
1777,NaN,NaN,NaN,NaN,GCA_000204665.1,NaN,NaN,PRJNA65323,SAMN02603905,NaN,...,131567::2::1783272::1239::91061::1385::90964::...,cellular organisms::Bacteria::Bacillati::Bacil...,cellular organisms; Bacteria; Bacillati; Bacil...,NaN,NaN,54.0,NaN,NaN,NaN,Staphylococcus aureus
2106,NaN,NaN,Resistant,Computational Prediction,GCA_902172305.1,NaN,NaN,PRJEB657,SAMEA1695737,NaN,...,131567::2::3379134::1224::1236::72274::135621:...,cellular organisms::Bacteria::Pseudomonadati::...,cellular organisms; Bacteria; Pseudomonadati; ...,NaN,NaN,64.0,NaN,NaN,NaN,Pseudomonas aeruginosa
2107,NaN,NaN,NaN,NaN,GCA_000408865.1,allpaths v. R40459,NaN,PRJNA66135,SAMN02596970,NaN,...,131567::2::3379134::1224::1236::72274::135621:...,cellular organisms::Bacteria::Pseudomonadati::...,cellular organisms; Bacteria; Pseudomonadati; ...,NaN,NaN,65.0,NaN,NaN,NaN,Pseudomonas aeruginosa


In [33]:
len(bv_brc_to_parq)

135137

In [35]:
len(bv_brc_df[['genome.isolation_source', 'genome.host_name']].drop_duplicates())

6596

In [36]:
len(bv_brc_df['genome.biosample_accession'].drop_duplicates())

95493